# 🧹 Prétraitement du dataset IMDB

## 🎯 Objectif
Transformer les critiques brutes (textes avec balises HTML) en séquences numériques exploitables par un modèle de deep learning.

## 📥 Entrée
- `data/imdb_train.csv` et `data/imdb_test.csv` (textes bruts + labels)

## 📤 Sortie
- `data/X_train.npy`, `data/X_test.npy` (séquences encodées)
- `data/y_train.npy`, `data/y_test.npy` (labels)
- `data/vocab.pkl` (dictionnaire mot → index)

## ⚙️ Étapes du prétraitement
1. Nettoyage du texte
   - Mise en minuscules
   - Suppression des balises HTML (`<br />`)
   - Suppression de la ponctuation et des chiffres
2. Tokenisation (découpe par espaces)
3. Construction du vocabulaire (top 20 000 mots)
4. Encodage des textes en indices
5. Padding / troncature à 200 mots

## 📊 Paramètres
| Paramètre | Valeur |
|-----------|--------|
| Vocabulaire | 20 000 mots |
| Longueur max | 200 mots |
| Token spécial PAD | index 0 |
| Token spécial UNK | index 1 |

## 📚 Table des matières
- [Imports et chargement](#imports-et-chargement)
- [Nettoyage du texte](#nettoyage-du-texte)
- [Construction du vocabulaire](#construction-du-vocabulaire)
- [Création du vocabulaire (top 20 000 mots)](#création-du-vocabulaire-top-20-000-mots)
- [Encodage + Padding](#encodage--padding)
- [Sauvegarde](#sauvegarde)

 #  Imports et chargement

In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter

# Chargement des données
train_df = pd.read_csv("data/imdb_train.csv")
test_df  = pd.read_csv("data/imdb_test.csv")

print(f"✅ Train : {len(train_df):,} | Test : {len(test_df):,}")

✅ Train : 25,000 | Test : 25,000


# Nettoyage du texte 

In [2]:
def clean_text(text):
    text = text.lower()                          # minuscules
    text = re.sub(r"<br\s*/?>", " ", text)       # supprime les <br>
    text = re.sub(r"[^a-z\s]", "", text)         # garde seulement les lettres
    text = re.sub(r"\s+", " ", text).strip()     # espaces multiples
    return text

# Application sur train et test
train_df["text_clean"] = train_df["text"].apply(clean_text)
test_df["text_clean"]  = test_df["text"].apply(clean_text)

# Vérification
print("AVANT :", train_df["text"].iloc[0][:150])
print()
print("APRÈS :", train_df["text_clean"].iloc[0][:150])

AVANT : I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard th

APRÈS : i rented i am curiousyellow from my video store because of all the controversy that surrounded it when it was first released in i also heard that at f


## une validation après nettoyage

In [3]:
# Après nettoyage, vérifier qu'aucun texte n'est vide
empty_train = (train_df["text_clean"].str.len() == 0).sum()
if empty_train > 0:
    print(f"⚠️ Attention : {empty_train} textes vides dans le train")

# Construction du vocabulaire Tokenisation 

In [4]:
# Tokenisation simple : découpe par espaces
def tokenize(text):
    return text.split()

# Compter tous les mots du train
all_words = []
for text in train_df["text_clean"]:
    all_words.extend(tokenize(text))

counter = Counter(all_words)
print(f"Nombre de mots uniques : {len(counter):,}")
print(f"Top 10 mots : {counter.most_common(10)}")

Nombre de mots uniques : 108,661
Top 10 mots : [('the', 334788), ('and', 162270), ('a', 161989), ('of', 145382), ('to', 135116), ('is', 107042), ('in', 93130), ('it', 78107), ('i', 75750), ('this', 75439)]


## statistiques après tokenisation

In [5]:
# Longueur moyenne après tokenisation
train_tokens_len = [len(tokenize(t)) for t in train_df["text_clean"]]
print(f"Longueur moyenne : {np.mean(train_tokens_len):.1f} mots")
print(f"Longueur max      : {max(train_tokens_len)} mots")
print(f"Quantile 95%      : {np.percentile(train_tokens_len, 95):.0f} mots")

Longueur moyenne : 229.4 mots
Longueur max      : 2450 mots
Quantile 95%      : 587 mots


# Création du vocabulaire (top 20 000 mots)

In [6]:
VOCAB_SIZE = 20000   # on garde les 20 000 mots les plus fréquents
MAX_LEN    = 200     # longueur max de chaque séquence (en mots)

# Tokens spéciaux
PAD_TOKEN = "<PAD>"   # rembourrage
UNK_TOKEN = "<U
NK>"   # mot inconnu

# Vocabulaire : index 0 = PAD, index 1 = UNK
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Taille du vocabulaire : {len(vocab):,}")
print(f"Exemples : the={vocab.get('the')}  "
      f"film={vocab.get('film')}  "
      f"good={vocab.get('good')}")

Taille du vocabulaire : 20,000
Exemples : the=2  film=19  good=49


#  Encodage + Padding 

In [7]:
def encode_and_pad(text, vocab, max_len):
    tokens = tokenize(text)[:max_len]           # on tronque si trop long
    ids = [vocab.get(t, 1) for t in tokens]     # 1 = UNK si mot inconnu
    # Padding à droite si trop court
    ids = ids + [0] * (max_len - len(ids))
    return ids

# Application
X_train = np.array([encode_and_pad(t, vocab, MAX_LEN) for t in train_df["text_clean"]])
X_test  = np.array([encode_and_pad(t, vocab, MAX_LEN) for t in test_df["text_clean"]])
y_train = train_df["label"].values
y_test  = test_df["label"].values

print(f"X_train shape : {X_train.shape}  ← doit être (25000, 200)")
print(f"X_test  shape : {X_test.shape}   ← doit être (25000, 200)")
print(f"y_train shape : {y_train.shape}")
print()
print("Exemple d'encodage (premiers 15 indices) :")
print(X_train[0][:15])

X_train shape : (25000, 200)  ← doit être (25000, 200)
X_test  shape : (25000, 200)   ← doit être (25000, 200)
y_train shape : (25000,)

Exemple d'encodage (premiers 15 indices) :
[  10 1541   10  237    1   35   55  390 1125   83    5   31    2 6940
   12]


## Sauvegarde

In [8]:
# Sauvegarde de tout pour les prochains notebooks
np.save("data/X_train.npy", X_train)
np.save("data/X_test.npy",  X_test)
np.save("data/y_train.npy", y_train)
np.save("data/y_test.npy",  y_test)

with open("data/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

print("✅ Fichiers sauvegardés dans data/ :")
print("   → X_train.npy  X_test.npy")
print("   → y_train.npy  y_test.npy")
print("   → vocab.pkl")
print()
print(f"   VOCAB_SIZE = {VOCAB_SIZE:,}")
print(f"   MAX_LEN    = {MAX_LEN}")
print()
print("✅ Étape 2 terminée ! Dis 'étape 3' pour continuer.")

✅ Fichiers sauvegardés dans data/ :
   → X_train.npy  X_test.npy
   → y_train.npy  y_test.npy
   → vocab.pkl

   VOCAB_SIZE = 20,000
   MAX_LEN    = 200

✅ Étape 2 terminée ! Dis 'étape 3' pour continuer.


## Résume

## ✅ Résumé du prétraitement

- **Textes nettoyés** : suppression HTML, minuscules, lettres uniquement
- **Vocabulaire** : 20 000 mots les plus fréquents (couvre ~95% des occurrences)
- **Séquences** : 25 000 train / 25 000 test, chacune de 200 mots (pad/trunc)
- **Fichiers sauvegardés** : X_train, X_test, y_train, y_test, vocab.pkl

✅ Prêt pour l'étape 3 : Modélisation